# Can the Baby GOAT Surpass the GOAT?
## 03 — Era-Adjusted Quarterback Performance

**Research question.** How did Tom Brady and Patrick Mahomes perform relative to the NFL environment of their respective seasons during their **first 131 regular-season starts**?

**Unit of comparison:** QB-attributed Expected Points Added (QB EPA) on eligible dropbacks (passes, sacks, and scrambles). The validated upstream pipeline restores historically tagged scrambles that were not flagged as dropbacks in the original PBP data.

This notebook is an **observational, descriptive comparison**, not an estimate of individual talent or causal QB impact. It does not assume the two quarterbacks played in identical team situations.

### Definitions

For quarterback $q$ in game $g$ of season $s$:

$$\text{Absolute QB EPA/DB}_{g}=\frac{\sum_{i\in g}\mathrm{QB\ EPA}_i}{N_g}$$

$$\text{Season-relative QB EPA/DB}_{g}=\text{Absolute QB EPA/DB}_g-\mu_s$$

where $N_g$ is the number of usable eligible dropbacks and $\mu_s$ is the NFL **dropback-weighted** mean QB EPA for season $s$.

We aggregate the first 131 starts by **total dropbacks**, not by taking an unweighted mean of game-level rates:

$$\text{Relative EPA/DB}_{131}=\frac{\sum_g(\mathrm{QB\ EPA}_g-N_g\mu_{s(g)})}{\sum_g N_g}$$

**Also included:** a season-standardized effect size, $(\mathrm{QB\ EPA/DB}-\mu_s)/\sigma_s$, where $\sigma_s$ is the **league play-level** standard deviation of QB EPA in that season. It is *not* a QB-rank z-score, a sampling z-statistic, or a QB percentile.

**Also included:** qualified-QB season percentiles computed against a season-specific cohort of eligible quarterbacks. These ranks describe full-season QB performance; when the scope differs from the selected 131 starts, it is explicitly flagged. Three-year relative EPA/DB is included as a final **league-baseline smoothing sensitivity analysis**. At study-era boundaries, two-year windows are used; partial 2026 is explicitly provisional.

## 1. Environment and validated sources

Run from the repository root with `jupyter lab`, or open the notebook from `notebooks/`. All inputs are **read-only** and generated by the validated processing scripts. Charts will be saved under `reports/figures/`.

Required inputs:

- `data/processed/nfl_season_dropback_baselines.parquet`
- `data/processed/qb_game_era_adjusted.parquet`
- `data/processed/qb_study_season_era_summary.parquet`
- `data/processed/qb_first_131_era_summary.parquet`

To rebuild inputs: run `scripts/build_nfl_epa_baselines.py` and then `scripts/build_qb_era_adjusted_stats.py`.

**Three-year extension inputs (after running `scripts/build_three_year_relative_epa.py`):**
- `data/processed/nfl_three_year_dropback_baselines.parquet`
- `data/processed/qb_game_three_year_relative.parquet`
- `data/processed/qb_study_season_three_year_summary.parquet`
- `data/processed/qb_first_131_three_year_summary.parquet`


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from IPython.display import Markdown, display

# Team color palette from the established descriptive-analysis visuals.
COLORS = {
    "Tom Brady": "#002244",       # New England Patriots
    "Patrick Mahomes": "#E31837", # Kansas City Chiefs
}
ORDER = ["Tom Brady", "Patrick Mahomes"]
INK = "#17202A"
SUBTLE = "#64748B"
GRID = "#E2E8F0"

# Detect the repository root whether Jupyter starts in root/ or notebooks/.
search_roots = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in search_roots if (p / "data/processed/qb_game_era_adjusted.parquet").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/qb_game_era_adjusted.parquet. "
        "Open Jupyter from the project root and run the data pipeline first."
    )

PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "reports" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

paths = {
    "games": PROCESSED / "qb_game_era_adjusted.parquet",
    "seasons": PROCESSED / "qb_study_season_era_summary.parquet",
    "career": PROCESSED / "qb_first_131_era_summary.parquet",
    "league": PROCESSED / "nfl_season_dropback_baselines.parquet",
}
missing_files = [str(p) for p in paths.values() if not p.is_file()]
if missing_files:
    raise FileNotFoundError("Missing validated inputs:\n" + "\n".join(missing_files))

games = pl.read_parquet(paths["games"])
seasons = pl.read_parquet(paths["seasons"])
career = pl.read_parquet(paths["career"])
league = pl.read_parquet(paths["league"])

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 15,
    "axes.labelsize": 11,
    "axes.edgecolor": "#CBD5E1",
    "axes.labelcolor": INK,
    "text.color": INK,
    "xtick.color": SUBTLE,
    "ytick.color": SUBTLE,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

print(f"Project: {ROOT}")
print(f"Games: {games.height} | Study-season rows: {seasons.height} | "
      f"Career rows: {career.height} | League seasons: {league.height}")

## 2. Quality gates — fail before interpretation

These checks verify the research sample, unique keys, denominators, baseline coverage, and reconciliation between game and career totals. **Do not remove a failing assertion simply to display the figures.** Check the upstream processing if anything changes.

In [ ]:
REQUIRED_GAMES = {
    "player_id", "player_name", "game_id", "start_no", "season",
    "usable_dropbacks", "qb_epa_sum", "qb_epa_per_dropback",
    "relative_qb_epa_total", "relative_qb_epa_per_dropback",
    "league_qb_epa_per_dropback", "qb_epa_success_rate",
    "relative_qb_epa_success_rate_pp",
}
REQUIRED_SEASONS = {
    "player_id", "player_name", "season", "starts", "usable_dropbacks",
    "qb_epa_per_dropback", "relative_qb_epa_per_dropback",
}
REQUIRED_CAREER = {
    "player_id", "player_name", "starts", "usable_dropbacks",
    "qb_epa_sum", "relative_qb_epa_total", "qb_epa_per_dropback",
    "relative_qb_epa_per_dropback", "expected_league_epa_per_dropback",
    "relative_qb_epa_success_rate_pp",
}
REQUIRED_LEAGUE = {
    "season", "baseline_scope", "usable_dropbacks",
    "league_qb_epa_per_dropback", "league_qb_epa_std",
}
for name, data, required in [
    ("games", games, REQUIRED_GAMES),
    ("study seasons", seasons, REQUIRED_SEASONS),
    ("career", career, REQUIRED_CAREER),
    ("league", league, REQUIRED_LEAGUE),
]:
    missing_cols = required - set(data.columns)
    assert not missing_cols, f"{name}: missing columns {sorted(missing_cols)}"

assert games.height == 262, "Expected exactly 262 starts"
assert career.height == 2, "Expected two quarterbacks"
assert league.height == 20, "Expected 20 NFL season baselines"
assert games.select(pl.struct("player_id", "game_id").n_unique()).item() == 262
assert league["season"].n_unique() == 20
assert sorted(games["season"].unique().to_list()) == sorted(league["season"].to_list())
assert set(games["player_name"].unique().to_list()) == set(ORDER)
assert not games.filter(pl.col("usable_dropbacks") <= 0).height
assert not league.filter(pl.col("league_qb_epa_std") <= 0).height

for player in ORDER:
    segment = games.filter(pl.col("player_name") == player).sort("start_no")
    assert segment.height == 131, f"{player}: expected 131 starts"
    assert segment["start_no"].to_list() == list(range(1, 132))
    row = career.filter(pl.col("player_name") == player).row(0, named=True)
    assert row["starts"] == 131
    assert int(segment["usable_dropbacks"].sum()) == int(row["usable_dropbacks"])
    assert np.isclose(float(segment["qb_epa_sum"].sum()), float(row["qb_epa_sum"]), atol=1e-6)
    assert np.isclose(float(segment["relative_qb_epa_total"].sum()),
                      float(row["relative_qb_epa_total"]), atol=1e-6)
    observed = float(row["relative_qb_epa_total"]) / float(row["usable_dropbacks"])
    assert np.isclose(observed, float(row["relative_qb_epa_per_dropback"]), atol=1e-10)

# All 2026 NFL context is explicitly partial, ending with the study date.
assert league.filter(pl.col("season") == 2026)["baseline_scope"].item() == "through_study_date"
assert league.filter(pl.col("season") != 2026).filter(
    pl.col("baseline_scope") != "full_regular_season"
).height == 0

print("PASS: 262 unique starts, 131 per QB, 20 matching baselines, valid denominators, reconciled career totals.")

## 3. Figure 1 — What was the league environment?

This figure shows the **dropback-weighted NFL average** for each study season, not the performance of either quarterback. The two panels deliberately separate eras instead of drawing an artificial line across the missing 2011–2016 period.

**Interpretation prompts:** Which seasons had the highest/lowest league averages? Did the average rise monotonically over time? How should we treat the provisional 2026 baseline?

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True, layout="constrained")
for ax, years, subtitle in zip(
    axs,
    [range(2001, 2011), range(2017, 2027)],
    ["Brady study era · 2001–2010", "Mahomes study era · 2017–2026"],
):
    table = league.filter(pl.col("season").is_in(list(years))).sort("season")
    x = np.asarray(table["season"].to_list(), dtype=int)
    y = np.asarray(table["league_qb_epa_per_dropback"].to_list(), dtype=float)
    ax.plot(x, y, color=INK, linewidth=2.1, alpha=0.8, zorder=2)
    ax.scatter(x[:-1], y[:-1], s=45, facecolor="white", edgecolor=INK,
               linewidth=1.6, zorder=3)
    # Plot the final year explicitly; 2026 uses a hollow marker and a note.
    if x[-1] == 2026:
        ax.scatter([x[-1]], [y[-1]], s=95, facecolor="white",
                   edgecolor=COLORS["Patrick Mahomes"], linewidth=2.3, zorder=4)
        ax.annotate("2026: partial", xy=(x[-1], y[-1]), xytext=(-65, 18),
                    textcoords="offset points", fontsize=9, color=COLORS["Patrick Mahomes"],
                    arrowprops=dict(arrowstyle="-", color=COLORS["Patrick Mahomes"], lw=0.9))
    else:
        ax.scatter([x[-1]], [y[-1]], s=45, facecolor="white", edgecolor=INK,
                   linewidth=1.6, zorder=3)
    ax.axhline(0, color=SUBTLE, linewidth=1, linestyle="--")
    ax.set_xticks(x[::2])
    ax.set_title(subtitle, loc="left", fontsize=12, weight="semibold")
    ax.grid(axis="y", color=GRID, linewidth=0.9)
    ax.spines[["right", "top"]].set_visible(False)
    ax.set_xlabel("NFL season")
axs[0].set_ylabel("League QB EPA per dropback")
fig.suptitle("NFL Passing Environment | 2001–2010 and 2017–2026",
             fontsize=16, weight="bold")
fig.savefig(FIGURES / "04_nfl_era_baselines.png", dpi=220, bbox_inches="tight")
plt.show()

## 4. Figure 2 — The first 131 starts: absolute vs season-relative EPA

Every headline number is **dropback-weighted**. The expected NFL value for each QB is itself weighted according to *that QB's distribution of dropbacks across seasons*. It is generally not identical to the unweighted average of league season means.

The colored mark is observed QB EPA/DB, the gray mark is the matching league expectation, and the difference between them is **season-relative EPA/DB**. No inference about individual talent follows from these descriptive differences alone.

In [ ]:
rows = {r["player_name"]: r for r in career.to_dicts()}
lines = [
    "| Quarterback | Dropbacks | Absolute QB EPA/DB | Era-matched NFL EPA/DB | Relative QB EPA/DB | Relative success rate (pp) |",
    "|:--|--:|--:|--:|--:|--:|",
]
for player in ORDER:
    r = rows[player]
    lines.append(
        f"| {player} | {int(r['usable_dropbacks']):,} | "
        f"{r['qb_epa_per_dropback']:+.4f} | "
        f"{r['expected_league_epa_per_dropback']:+.4f} | "
        f"{r['relative_qb_epa_per_dropback']:+.4f} | "
        f"{r['relative_qb_epa_success_rate_pp']:+.2f} |"
    )
display(Markdown("\n".join(lines)))

values = [
    float(rows[p][k])
    for p in ORDER
    for k in ["qb_epa_per_dropback", "expected_league_epa_per_dropback"]
]
lo, hi = min(values), max(values)
span = max(hi - lo, 0.08)
fig, ax = plt.subplots(figsize=(11.4, 4.3), layout="constrained")
for i, player in enumerate(ORDER):
    r = rows[player]
    qb = float(r["qb_epa_per_dropback"])
    baseline = float(r["expected_league_epa_per_dropback"])
    color = COLORS[player]
    y = len(ORDER) - i
    ax.plot([baseline, qb], [y, y], color=color, alpha=0.40, linewidth=5,
            solid_capstyle="round", zorder=1)
    ax.scatter([baseline], [y], s=85, facecolor="white", edgecolor=SUBTLE,
               linewidth=2, zorder=3)
    ax.scatter([qb], [y], s=135, facecolor=color, edgecolor="white",
               linewidth=1, zorder=4)
    ax.annotate(f"League {baseline:+.3f}", (baseline, y), xytext=(0, 15),
                textcoords="offset points", ha="center", fontsize=9, color=SUBTLE)
    ax.annotate(f"QB {qb:+.3f}", (qb, y), xytext=(0, -22),
                textcoords="offset points", ha="center", fontsize=10,
                color=color, weight="bold")
ax.axvline(0, color=SUBTLE, linestyle="--", linewidth=1)
ax.set_xlim(lo - 0.35 * span, hi + 0.35 * span)
ax.set_yticks([2, 1], ORDER)
ax.set_ylim(0.45, 2.55)
ax.grid(axis="x", color=GRID, linewidth=0.9)
ax.set_axisbelow(True)
ax.spines[["right", "top", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0, labelsize=12)
ax.set_xlabel("QB EPA per eligible dropback")
ax.set_title("131 Starts | QB Efficiency vs Era-Matched NFL Expectation",
             fontsize=15, weight="bold", loc="left", pad=17)
fig.savefig(FIGURES / "05_qb_vs_era_baseline_131.png", dpi=220, bbox_inches="tight")
plt.show()

leader = max(ORDER, key=lambda p: float(rows[p]["relative_qb_epa_per_dropback"]))
follower = next(p for p in ORDER if p != leader)
delta = float(rows[leader]["relative_qb_epa_per_dropback"]) - float(rows[follower]["relative_qb_epa_per_dropback"])
print(f"Higher observed era-relative QB EPA/DB: {leader} (difference: {delta:+.4f} EPA/DB).")
print("This is a descriptive difference, not a QB-only causal effect or significance test.")

## 5. Figure 3 — Which study seasons contributed to the result?

Season bars represent **only the starts selected within the first-131-start sample**, not the player's entire season. Therefore, a one-start season can be extreme without carrying much weight in the 131-start aggregate.

Key small-sample cases: Brady **2008 (1 start)**, Brady **2010 (4 starts)**, Mahomes **2017 (1 start)**, and Mahomes **2026 (5 starts, partial NFL baseline)**.

**Interpretation prompts:** Which stretches were above/below their contemporary league mean? Do unusually high/low bars have many dropbacks? How different is the 131-start result from any single season?

In [ ]:
season_rows = seasons.to_dicts()
max_abs = max(abs(float(r["relative_qb_epa_per_dropback"])) for r in season_rows)
scale = max(max_abs * 1.18, 0.06)
fig, axs = plt.subplots(2, 1, figsize=(13, 8.8), sharey=True, layout="constrained")
for ax, player in zip(axs, ORDER):
    data = sorted((r for r in season_rows if r["player_name"] == player),
                  key=lambda r: int(r["season"]))
    xs = np.arange(len(data))
    ys = np.array([float(r["relative_qb_epa_per_dropback"]) for r in data])
    bars = ax.bar(xs, ys, width=0.69, color=COLORS[player], alpha=0.91)
    for bar, row, val in zip(bars, data, ys):
        if int(row["starts"]) <= 5:
            ax.annotate(f"{row['starts']} start{'s' if row['starts'] != 1 else ''}",
                        (bar.get_x() + bar.get_width()/2, val),
                        xytext=(0, 6 if val >= 0 else -7), textcoords="offset points",
                        ha="center", va="bottom" if val >= 0 else "top",
                        fontsize=9, color=INK)
        if int(row["season"]) == 2026:
            bar.set_edgecolor(INK)
            bar.set_linewidth(1.5)
            bar.set_hatch("//")
    ax.set_xticks(xs, [str(r["season"]) for r in data])
    ax.axhline(0, color=INK, linewidth=1)
    ax.set_ylim(-scale, scale)
    ax.set_ylabel("Relative QB EPA/DB")
    ax.set_title(player, loc="left", weight="bold", color=COLORS[player])
    ax.grid(axis="y", color=GRID, linewidth=0.9)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
axs[1].set_xlabel("NFL season (study starts only)")
fig.suptitle("Season-Relative QB EPA/DB | Selected Career Starts",
             fontsize=17, weight="bold")
fig.savefig(FIGURES / "06_season_relative_qb_epa.png", dpi=220, bbox_inches="tight")
plt.show()

## 6. Figure 4 — How did the era-relative advantage evolve over 131 starts?

The cumulative series uses **all dropbacks up to a given career start**:

$$\mathrm{Cumulative\ relative\ EPA/DB}(k)=\frac{\sum_{g=1}^{k}\mathrm{Relative\ EPA}_g}{\sum_{g=1}^{k} N_g}$$

A rolling **16-start** series shows local variation using the most recent 16 starts, also weighted by the number of dropbacks. This is a *game-window*, not a three-year season-relative benchmark. Early cumulative observations are unstable because they are based on small samples.

**Interpretation prompts:** Does the advantage persist throughout the sample? Are there different phases? Do high peaks or troughs depend on individual games or sustained runs?

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14.5, 5.2), sharex=True, layout="constrained")
WINDOW = 16
for player in ORDER:
    segment = games.filter(pl.col("player_name") == player).sort("start_no")
    start = np.asarray(segment["start_no"].to_list(), dtype=int)
    n = np.asarray(segment["usable_dropbacks"].to_list(), dtype=float)
    delta = np.asarray(segment["relative_qb_epa_total"].to_list(), dtype=float)
    cumulative = np.cumsum(delta) / np.cumsum(n)
    rolling = np.full(len(n), np.nan)
    for end in range(WINDOW - 1, len(n)):
        left = end - WINDOW + 1
        rolling[end] = delta[left:end+1].sum() / n[left:end+1].sum()
    for ax, series, name in [
        (axs[0], cumulative, "Cumulative"),
        (axs[1], rolling, "Rolling"),
    ]:
        ax.plot(start, series, color=COLORS[player], linewidth=2.35,
                label=player, solid_capstyle="round")
    axs[0].scatter([131], [cumulative[-1]], color=COLORS[player], s=60, zorder=3)

for ax in axs:
    ax.axhline(0, color=INK, linestyle="--", linewidth=1)
    ax.set_xlim(1, 131)
    ax.set_xticks([1, 25, 50, 75, 100, 131])
    ax.grid(axis="y", color=GRID)
    ax.set_xlabel("Career regular-season start number")
    ax.set_ylabel("Relative QB EPA per dropback")
    ax.spines[["top", "right"]].set_visible(False)
axs[0].set_title("Cumulative through each start", loc="left", weight="bold")
axs[1].set_title(f"Trailing {WINDOW}-start weighted window", loc="left", weight="bold")
axs[1].legend(loc="best", frameon=False)
fig.suptitle("Career Evolution | Era-Relative QB Efficiency", fontsize=17, weight="bold")
fig.savefig(FIGURES / "07_era_relative_career_evolution.png", dpi=220, bbox_inches="tight")
plt.show()

## 7. Figure 5 — Season-normalized EPA (dropback-SD units)

As an **additional descriptive lens**, divide the season-relative EPA/DB by the standard deviation of *individual dropback EPA values* across that NFL season:

$$z^{\mathrm{playSD}}_g=\frac{\mathrm{QB\ EPA/DB}_g-\mu_s}{\sigma_s^{\mathrm{dropback}}}$$

For the first 131 starts, we average game-level $z^{\mathrm{playSD}}$ values **weighted by usable dropbacks**. This accounts for changes in the *spread of play-level outcomes* across seasons.

**Do not interpret this as:** an NFL quarterback percentile, the number of standard deviations by which a QB beats other qualified QBs, or a z-test of statistical significance. Those require a separate **QB-level** reference distribution and qualification rule.

In [ ]:
# qb_game_era_adjusted contains the season mean, but not the play-level SD.
normalized_games = games.join(
    league.select("season", "league_qb_epa_std"),
    on="season", how="left", validate="m:1",
)
assert not normalized_games.filter(pl.col("league_qb_epa_std").is_null()).height
normalized_games = normalized_games.with_columns(
    (
        pl.col("relative_qb_epa_per_dropback") / pl.col("league_qb_epa_std")
    ).alias("relative_epa_in_dropback_sd")
)

normalized_seasons = (
    normalized_games.group_by("player_name", "season")
    .agg(
        pl.col("usable_dropbacks").sum().alias("usable_dropbacks"),
        (
            (pl.col("relative_epa_in_dropback_sd") * pl.col("usable_dropbacks")).sum()
            / pl.col("usable_dropbacks").sum()
        ).alias("standardized_relative_epa")
    )
    .sort("player_name", "season")
)
normalized_career = (
    normalized_games.group_by("player_name")
    .agg(
        pl.col("usable_dropbacks").sum().alias("usable_dropbacks"),
        (
            (pl.col("relative_epa_in_dropback_sd") * pl.col("usable_dropbacks")).sum()
            / pl.col("usable_dropbacks").sum()
        ).alias("standardized_relative_epa")
    )
    .sort("player_name")
)

display(normalized_career)

fig, axs = plt.subplots(2, 1, figsize=(13, 8.2), sharey=True, layout="constrained")
normalized_rows = normalized_seasons.to_dicts()
lim = max(0.05, max(abs(float(r["standardized_relative_epa"])) for r in normalized_rows) * 1.20)
for ax, player in zip(axs, ORDER):
    data = sorted((r for r in normalized_rows if r["player_name"] == player),
                  key=lambda r: int(r["season"]))
    xs = np.arange(len(data))
    ys = np.array([float(r["standardized_relative_epa"]) for r in data])
    ax.bar(xs, ys, color=COLORS[player], width=0.68)
    ax.axhline(0, color=INK, linewidth=1)
    ax.set_ylim(-lim, lim)
    ax.set_xticks(xs, [str(r["season"]) for r in data])
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("Relative EPA/DB ÷ league play SD")
    ax.set_title(player, color=COLORS[player], weight="bold", loc="left")
    ax.spines[["top", "right", "left"]].set_visible(False)
axs[1].set_xlabel("Study season")
fig.suptitle("Season-Standardized QB EPA | Descriptive Play-Level SD Units",
             fontsize=16, weight="bold")
fig.savefig(FIGURES / "08_season_normalized_qb_epa.png", dpi=220, bbox_inches="tight")
plt.show()

## 8. Relative success rate — a secondary robustness lens

EPA averages can be influenced by a small number of very high- or low-value plays. The **positive-QB-EPA play rate** provides a complementary descriptive perspective. Its era-adjusted version is expressed in **percentage points**, not EPA units.

This rate does **not** replace EPA/DB: it discards information about play magnitude, and a QB can score well on one measure and less well on the other.

In [ ]:
summary_rows = {r["player_name"]: r for r in career.to_dicts()}
lines = [
    "| Quarterback | QB EPA/DB | Season-relative EPA/DB | Season-normalized effect | Relative success rate |",
    "|:--|--:|--:|--:|--:|",
]
normalized_lookup = {r["player_name"]: r for r in normalized_career.to_dicts()}
for player in ORDER:
    r = summary_rows[player]
    z = normalized_lookup[player]["standardized_relative_epa"]
    lines.append(
        f"| {player} | {r['qb_epa_per_dropback']:+.4f} | "
        f"{r['relative_qb_epa_per_dropback']:+.4f} | "
        f"{z:+.4f} | {r['relative_qb_epa_success_rate_pp']:+.2f} pp |"
    )
display(Markdown("\n".join(lines)))

## 9. Qualified QB percentile — place each season in the QB distribution

The league baseline is the **dropback-weighted mean of all NFL plays**. A *qualified QB percentile* instead ranks the **QB-season averages of qualifying players**, with each qualified quarterback receiving one rank. These are different questions.

**Qualification (research rule):** at least **200 usable dropbacks** in a full regular season, prorated according to league games available through the 2026 study cutoff. We also audit alternative thresholds of 150 and 250. These are not official NFL eligibility standards.

Percentile is `100 × (N − descending_midrank)/(N − 1)` (higher is better). The best QB has 100; the lowest has 0. It is **not** a play-level z-score, a statistical confidence level, or a claim about causal QB ability.

**Scope warning:** These distributions describe *entire player-seasons*, not necessarily just the player's 131 selected starts. A study-aligned percentile is shown only when total usable dropbacks **and** total EPA match the selected starts. The complete-season percentile is retained and labeled separately when the samples differ (e.g. Brady in 2010).

In [ ]:
from matplotlib.lines import Line2D

qualified_path = PROCESSED / "nfl_qualified_qb_season_epa.parquet"
study_pct_path = PROCESSED / "study_qb_qualified_percentiles.parquet"
sensitivity_path = ROOT / "reports/validation/qualified_qb_threshold_sensitivity.csv"
for p in (qualified_path, study_pct_path, sensitivity_path):
    if not p.is_file():
        raise FileNotFoundError(
            f"Missing {p}. First run: python3 scripts/build_qualified_qb_percentiles.py"
        )

qualified = pl.read_parquet(qualified_path)
study_pct = pl.read_parquet(study_pct_path)
sensitivity = pl.read_csv(sensitivity_path)
assert study_pct.height == 20
assert qualified['season'].n_unique() == 20
assert qualified.select(pl.struct('season', 'player_id').n_unique()).item() == qualified.height
assert not qualified.filter(pl.col('qualified_qb_percentile').is_null()).height
assert not qualified.filter((pl.col('qualified_qb_percentile') < 0) | (pl.col('qualified_qb_percentile') > 100)).height

lines = [
    '| QB | Season | Qualifying QBs | Rank | Full-season percentile | Study-aligned percentile | Sample status |',
    '|:--|--:|--:|--:|--:|--:|:--|',
]
for row in study_pct.sort('player_name', 'season').to_dicts():
    pct = row['qualified_qb_percentile']
    aligned = row['study_aligned_qualified_percentile']
    rank = row['rank_descending']
    lines.append(
        f"| {row['player_name']} | {row['season']} | "
        f"{row['qualified_qb_count'] if row['qualified_qb_count'] is not None else '—'} | "
        f"{f'{rank:.1f}' if rank is not None else '—'} | "
        f"{f'{pct:.1f}' if pct is not None else 'N/Q'} | "
        f"{f'{aligned:.1f}' if aligned is not None else 'N/A'} | "
        f"{row['interpretation_status']} |"
    )

display(Markdown('\n'.join(lines)))
print(f"Qualified QB-season observations: {qualified.height}")
print('Strictly study-aligned qualified seasons:',
      study_pct.filter(pl.col('study_aligned_qualified_percentile').is_not_null()).height)


### Figure 6 — Seasonal distributions of qualified QB EPA/dropback

Each gray point is one **qualified QB's season average EPA/dropback**, not an individual play. Colored points identify the research QB's *full-season* EPA/DB. Filled markers match the study-start population; hollow diamonds use a wider season population. Years without a qualified research QB have no colored marker.

**Read the vertical position as absolute QB EPA/DB; the percentile of that QB is shown separately below.** Gray points make it possible to judge whether the season was tightly packed or had a few very strong outliers.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9.6), layout='constrained', sharey=True)
all_epa = qualified['qb_epa_per_dropback'].to_numpy()
y_low, y_high = float(np.min(all_epa)), float(np.max(all_epa))
pad = max((y_high - y_low) * 0.10, 0.045)

for ax, player in zip(axes, ORDER):
    years = list(range(2001, 2011)) if player == 'Tom Brady' else list(range(2017, 2027))
    rng = np.random.default_rng(42)  # Deterministic visual jitter; no data changes.
    for yr in years:
        group = qualified.filter(pl.col('season') == yr)
        values = group['qb_epa_per_dropback'].to_numpy()
        xs = yr + rng.uniform(-0.23, 0.23, size=len(values))
        ax.scatter(xs, values, s=28, color='#95A2B3', alpha=0.5,
                   zorder=2, linewidths=0)
    focus = study_pct.filter(pl.col('player_name') == player).sort('season').to_dicts()
    for row in focus:
        if not row['qualified']:
            continue
        x, y = int(row['season']), float(row['fullseason_qb_epa_per_dropback'])
        if row['fullseason_matches_study_starts']:
            ax.scatter(x, y, color=COLORS[player], s=110, edgecolor='white',
                       linewidth=1.8, zorder=5)
        else:
            ax.scatter(x, y, facecolors='white', edgecolors=COLORS[player], marker='D',
                       s=98, linewidth=2.5, zorder=6)
    ax.set_xticks(years)
    ax.set_xlim(min(years)-0.6, max(years)+0.6)
    ax.set_ylim(y_low-pad, y_high+pad)
    ax.axhline(0, color=INK, linewidth=1, linestyle='--', alpha=0.55)
    ax.grid(axis='y', color=GRID, linewidth=0.9)
    ax.set_axisbelow(True)
    ax.set_ylabel('QB season EPA per dropback')
    ax.set_title(player, loc='left', color=COLORS[player], fontweight='bold')
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(axis='y', length=0)
    ax.set_xlabel('NFL season (2026 truncated at study date)' if player == 'Patrick Mahomes'
                  else 'NFL season')

legend = [
    Line2D([0], [0], linestyle='', marker='o', markerfacecolor='#95A2B3',
           markeredgecolor='none', markersize=8, label='Qualified QB season'),
    Line2D([0], [0], linestyle='', marker='o', color=INK, markersize=9,
           label='Study-aligned research QB'),
    Line2D([0], [0], linestyle='', marker='D', markerfacecolor='white',
           markeredgecolor=INK, markersize=8, label='Full season includes non-study plays'),
]
axes[0].legend(handles=legend, frameon=False, loc='upper right', ncol=1)
fig.suptitle('Qualified QB EPA/Dropback | Season-by-Season Peer Distributions',
             fontsize=17, weight='bold')
fig.savefig(FIGURES / '09_qualified_qb_season_distributions.png',
            dpi=240, bbox_inches='tight')
plt.show()


### Figure 7 — Percentile trajectory within qualified QB peer groups

A percentile near 90 means the QB ranked above roughly 90% of that season's qualifying QBs under the stated rank definition. **It does not mean 90% more EPA, nor a 90% likelihood of being the best player.**

- **Filled circle:** the full-season QB dropbacks are identical to the study subset, so the ranking can also describe the selected starts.
- **Hollow diamond:** the full-season QB percentile includes plays *outside* the selected 131 starts. It is **context only**, not a study-start percentile.
- **N/Q:** not qualified; no percentile is assigned. Neither a one-game start nor a small sample should be stretched into a season-long ranking.
- **2026:** provisional peer group through the study date, with a prorated cutoff.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8.4), layout='constrained', sharey=True)
for ax, player in zip(axes, ORDER):
    focus = study_pct.filter(pl.col('player_name') == player).sort('season').to_dicts()
    years = [int(r['season']) for r in focus]
    pcts = [float(r['qualified_qb_percentile']) if r['qualified_qb_percentile'] is not None
            else np.nan for r in focus]
    ax.plot(years, pcts, color=COLORS[player], linewidth=2.2, alpha=0.7)
    for r in focus:
        x = int(r['season'])
        value = r['qualified_qb_percentile']
        if value is None:
            ax.text(x, 5, 'N/Q', ha='center', fontsize=9, color=SUBTLE)
        elif r['fullseason_matches_study_starts']:
            ax.scatter(x, float(value), s=100, c=COLORS[player],
                       edgecolors='white', linewidths=1.2, zorder=5)
        else:
            ax.scatter(x, float(value), s=95, facecolors='white',
                       edgecolors=COLORS[player], marker='D', linewidths=2, zorder=6)
        if x == 2026:
            ax.text(x, 8, 'partial', ha='right', fontsize=9, color=SUBTLE, rotation=90)
    ax.set_ylim(-7, 108)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.set_xticks(years)
    ax.set_xlim(min(years)-0.5, max(years)+0.5)
    ax.grid(axis='y', color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_title(player, loc='left', color=COLORS[player], weight='bold')
    ax.set_ylabel('Qualified QB percentile')
    ax.set_xlabel('NFL season')
    ax.spines[['top','right','left']].set_visible(False)
    ax.tick_params(axis='y', length=0)
fig.suptitle('Where Each QB Ranked Among Qualified Peers', fontsize=17, weight='bold')
fig.savefig(FIGURES / '10_qualified_qb_percentiles.png',
            dpi=240, bbox_inches='tight')
plt.show()

# Sensitivity check: how far does each QB full-season percentile move when
# the qualification rule changes from 200 to 150 or 250 season dropbacks?
study_ids = study_pct.select('season', 'player_id', 'player_name')
sens_study = sensitivity.join(study_ids, on=['season', 'player_id'], how='inner')
sens_wide = sens_study.pivot(
    index=['season', 'player_id', 'player_name'],
    on='threshold_full_season',
    values='qualified_qb_percentile',
).sort('player_name', 'season')
print('Sensitivity of study QB full-season percentiles to qualification cutoffs:')
display(sens_wide)


### Qualified QB percentile table — season-by-season comparison

This compact table accompanies Figures 6 and 7. **Exact percentiles** are loaded from the validated `study_qb_qualified_percentiles.parquet` file; none are copied from a chart or rounded by eye.

Each column follows the player's own NFL seasons chronologically, rather than matching calendar years. The research uses the **full-season qualified-QB peer ranking**; years whose QB data include plays outside our selected starts are marked explicitly. Percentile 100 means the best-ranked qualified QB, not perfect performance.

In [ ]:
# Compare qualified QB percentiles side by side using exact season data.
# The percentages come from the processed percentile table, not chart estimates.
percentiles_by_qb = {
    player: (
        study_pct.filter(pl.col("player_name") == player)
        .sort("season")
        .to_dicts()
    )
    for player in ORDER
}

# Both players cover ten distinct NFL seasons in the research timeline.
assert [r["season"] for r in percentiles_by_qb["Tom Brady"]] == list(range(2001, 2011))
assert [r["season"] for r in percentiles_by_qb["Patrick Mahomes"]] == list(range(2017, 2027))


def format_season_percentile(row):
    """Display the year, exact computed percentile, and sample-scope markers."""
    pct = row["qualified_qb_percentile"]
    result = "N/Q" if pct is None else f"{pct:.1f}%"
    if pct is not None and not row["fullseason_matches_study_starts"]:
        result += "†"  # A full-season rank includes non-study QB plays.
    if int(row["season"]) == 2026:
        result += "‡"  # Partial NFL season through the study cutoff date.
    return f"**{int(row['season'])}** · {result}"

brady_rows = percentiles_by_qb["Tom Brady"]
mahomes_rows = percentiles_by_qb["Patrick Mahomes"]

percentile_table = [
    "| Tom Brady | Patrick Mahomes |",
    "|:--|:--|",
]
for brady, mahomes in zip(brady_rows, mahomes_rows):
    percentile_table.append(
        f"| {format_season_percentile(brady)} | "
        f"{format_season_percentile(mahomes)} |"
    )

percentile_table.extend([
    "",
    "**N/Q:** Does not meet the study's minimum qualifying dropbacks; no percentile is assigned.",
    "**†:** The full-season percentile includes dropbacks outside the 131 selected starts; use as seasonal context, not as a study-only rank.",
    "**‡:** The 2026 percentile is provisional, based on league data through October 18, 2026 and a prorated qualification threshold.",
    "Percentiles use the 200-dropback research threshold (prorated in 2026), "
    "with midrank treatment for ties; higher is better.",
])

display(Markdown("\n".join(percentile_table)))

### Qualification-threshold sensitivity — 150 / 200 / 250 dropbacks

The percentile depends on the qualifying-QB peer group. To test robustness, recompute Brady's and Mahomes's **full-season** percentiles with minimum qualification thresholds of **150, 200, and 250 usable dropbacks** (the main analysis uses 200). For partial 2026, all thresholds are prorated according to the number of NFL games included by the study cutoff.

A season can be **N/Q** at one cutoff but qualified at another. We do not treat N/Q as zero. **Range (pp)** is the largest minus the smallest percentile in percentage points, calculated only when the QB qualifies at **all three** cutoffs. A small range indicates stability *to this particular selection rule*, not statistical certainty. Percentiles marked as including non-study plays and those for partial 2026 retain the same scope limitations noted above.

The cell displays the comparison and exports an auditable study-player report to `reports/validation/qualified_qb_sensitivity_summary.csv`. It does not modify the validated EPA datasets.

In [ ]:
# Summarize the three qualification rules for the two study quarterbacks.
# The long-form source file was already generated by build_qualified_qb_percentiles.py.
# Run the notebook from the beginning so study_pct and sensitivity are loaded.
THRESHOLDS = (150, 200, 250)

sensitivity_lookup = {
    (int(r["season"]), str(r["player_id"]), int(r["threshold_full_season"])): r
    for r in sensitivity.to_dicts()
}

report_rows = []
for player in ORDER:
    for qb in study_pct.filter(pl.col("player_name") == player).sort("season").to_dicts():
        year, qb_id = int(qb["season"]), str(qb["player_id"])
        record = {
            "player_name": player,
            "season": year,
            "sample_status": qb["interpretation_status"],
        }
        percentiles = []
        for threshold in THRESHOLDS:
            match = sensitivity_lookup.get((year, qb_id, threshold))
            pct = float(match["qualified_qb_percentile"]) if match is not None else None
            record[f"percentile_{threshold}"] = pct
            record[f"peers_{threshold}"] = int(match["qualified_qb_count"]) if match is not None else None
            record[f"effective_min_{threshold}"] = int(match["threshold_effective"]) if match is not None else None
            if pct is not None:
                percentiles.append(pct)
        record["qualified_all_three"] = len(percentiles) == len(THRESHOLDS)
        record["percentile_range_pp"] = (
            max(percentiles) - min(percentiles)
            if record["qualified_all_three"] else None
        )
        report_rows.append(record)

assert len(report_rows) == 20, "Expected exactly 20 study QB-season records"

# Guardrail: the 200-DB column must reproduce the main validated results.
for record in report_rows:
    original = study_pct.filter(
        (pl.col("player_name") == record["player_name"])
        & (pl.col("season") == record["season"])
    ).row(0, named=True)["qualified_qb_percentile"]
    new = record["percentile_200"]
    assert (original is None and new is None) or (
        original is not None and new is not None and abs(float(original) - new) < 1e-9
    ), f"200-DB reference mismatch for {record['player_name']} {record['season']}"

sensitivity_report = pl.DataFrame(report_rows, infer_schema_length=None)
report_path = ROOT / "reports" / "validation" / "qualified_qb_sensitivity_summary.csv"
report_path.parent.mkdir(parents=True, exist_ok=True)
sensitivity_report.write_csv(report_path)

def show_pct(value):
    return "N/Q" if value is None else f"{value:.1f}%"

lines = [
    "| Quarterback | Season | Min 150 | Min 200 | Min 250 | Range (pp) |",
    "|:--|--:|--:|--:|--:|--:|",
]
for row in report_rows:
    span = row["percentile_range_pp"]
    lines.append(
        f"| {row['player_name']} | {row['season']} | "
        f"{show_pct(row['percentile_150'])} | {show_pct(row['percentile_200'])} | "
        f"{show_pct(row['percentile_250'])} | "
        f"{'N/A' if span is None else f'{span:.1f}'} |"
    )
display(Markdown("\n".join(lines)))

comparable = [r for r in report_rows if r["qualified_all_three"]]
changed_qualification = [
    r for r in report_rows
    if 0 < sum(r[f"percentile_{t}"] is not None for t in THRESHOLDS) < 3
]
if comparable:
    top_changes = sorted(comparable, key=lambda r: r["percentile_range_pp"], reverse=True)
    largest = top_changes[0]
    print(f"Comparable seasons: {len(comparable)}/20")
    print(f"Largest range: {largest['player_name']} {largest['season']} — "
          f"{largest['percentile_range_pp']:.2f} percentage points")
    print(f"Mean range: {sum(r['percentile_range_pp'] for r in comparable) / len(comparable):.2f} pp")
print(f"Qualification-status changes across thresholds: {len(changed_qualification)}")
for r in changed_qualification:
    print(f"  {r['player_name']} {r['season']}: " + ", ".join(
        f"{t}={show_pct(r[f'percentile_{t}'])}" for t in THRESHOLDS
    ))
print(f"Saved study-player sensitivity report: {report_path}")


## 10. Three-Year Relative EPA/Dropback — League baseline smoothing

**Research question.** Would the comparison between Brady and Mahomes change if we smoothed out unusually high or low NFL seasons instead of using only a single-year reference?

For study season $s$, the **league window mean** is a *dropback-weighted* aggregate of available adjacent seasons:

$$\mu_{W(s)}=\frac{\sum_{t\in W(s)}\mathrm{NFL\ QB\ EPA}_{t}}{\sum_{t\in W(s)}N_{t,\mathrm{valid}}}$$

For a study quarterback's game $g$ in season $s$:

$$\mathrm{ThreeYearRelativeEPA/DB}_g=\mathrm{QB\ EPA/DB}_g-\mu_{W(s)}$$

Across the **first 131 starts**, the game differences are weighted by the number of usable study QB dropbacks, not equally by game or year.

### Window coverage and methodological safeguards

- **Centered three full seasons:** 2007 uses 2006, 2007 and 2008; 2022 uses 2021, 2022 and 2023.
- **Era edges:** 2001, 2010, 2017 and 2025 use **two adjacent complete seasons** because a third complete season in the same study era is unavailable. Do not connect 2010 to 2017 or blend partial 2026 into the completed 2025 reference.
- **2026 only:** a **provisional trailing** window combines full 2024 and 2025 with 2026 **only through October 18**. It is not a conventional symmetric centered window.
- A smoother reference is an **alternative descriptive benchmark**, not a causal correction, QB ranking, or statistical significance test. Centered windows also use a following season and are therefore retrospective.

The implementation and exceptional cases are documented in `docs/three_year_relative_epa_methodology.md`.

In [ ]:
# Load the new validated outputs; do not recalculate EPA from raw play-by-play.
window_files = {
    "window_league": PROCESSED / "nfl_three_year_dropback_baselines.parquet",
    "window_games": PROCESSED / "qb_game_three_year_relative.parquet",
    "window_seasons": PROCESSED / "qb_study_season_three_year_summary.parquet",
    "window_career": PROCESSED / "qb_first_131_three_year_summary.parquet",
}
missing_window_files = [str(p) for p in window_files.values() if not p.is_file()]
if missing_window_files:
    raise FileNotFoundError(
        "First run python3 scripts/build_three_year_relative_epa.py\n" +
        "\n".join(missing_window_files)
    )
window_league = pl.read_parquet(window_files["window_league"])
window_games = pl.read_parquet(window_files["window_games"])
window_seasons = pl.read_parquet(window_files["window_seasons"])
window_career = pl.read_parquet(window_files["window_career"])

assert window_league.height == 20
assert window_games.height == games.height == 262
assert window_seasons.height == 20
assert window_career.height == 2
assert set(window_games["game_id"]) == set(games["game_id"])
assert window_games.select(pl.struct("player_id", "game_id").n_unique()).item() == 262
assert window_league.filter(pl.col("season") == 2025)["window_seasons"].item() == "2024,2025"
assert window_league.filter(pl.col("season") == 2026)["window_kind"].item() == "trailing_2_full_plus_partial"

for r in window_career.iter_rows(named=True):
    prior = career.filter(pl.col("player_id") == r["player_id"]).row(0, named=True)
    assert r["starts"] == prior["starts"] == 131
    assert r["usable_dropbacks"] == prior["usable_dropbacks"]
    assert np.isclose(r["qb_epa_per_dropback"], prior["qb_epa_per_dropback"], atol=1e-8)
    assert np.isclose(r["annual_relative_qb_epa_per_dropback"],
                      prior["relative_qb_epa_per_dropback"], atol=1e-8)

print("PASS: 20 NFL windows; 262 QB games; annual metrics unchanged; 2026 provisional.")
print(window_league.select("season", "window_seasons", "window_kind", "three_year_league_qb_epa_per_dropback"))

### Figure 8 — Annual NFL baseline vs the three-year window

The dark line shows the validated **annual** league EPA/dropback; the dashed accent line shows the **smoothed window** reference. The spread between the lines is precisely what can change the season-relative result. The final 2026 point includes partial-season data and is provisional.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5.0), sharey=True, layout="constrained")
for ax, player, era_years in zip(
    axs,
    ORDER,
    [range(2001, 2011), range(2017, 2027)],
):
    subset = window_league.filter(pl.col("season").is_in(list(era_years))).sort("season")
    xs = subset["season"].to_numpy()
    annual = subset["annual_league_qb_epa_per_dropback"].to_numpy()
    smooth = subset["three_year_league_qb_epa_per_dropback"].to_numpy()
    ax.plot(xs, annual, color=INK, marker="o", linewidth=2, label="Annual NFL EPA/DB")
    ax.plot(xs, smooth, color=COLORS[player], marker="s", linestyle="--",
            linewidth=2.2, label="Window-weighted NFL EPA/DB")
    ax.axhline(0, color=SUBTLE, linewidth=1, linestyle=":")
    ax.set_title(f"{player} study era", fontweight="bold", loc="left")
    ax.set_xticks(xs[::2])
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
    if player == "Patrick Mahomes":
        ax.axvspan(2025.65, 2026.35, color=COLORS[player], alpha=0.07)
        ax.annotate("2026: partial", xy=(2026, smooth[-1]),
                    xytext=(2022, max(smooth) + 0.016),
                    color=COLORS[player], fontsize=9,
                    arrowprops=dict(arrowstyle="-", color=COLORS[player]))
    ax.set_xlabel("NFL season")
axs[0].set_ylabel("League QB EPA per eligible dropback")
axs[0].legend(frameon=False, loc="best", fontsize=9)
fig.suptitle("Annual vs Three-Year NFL Dropback Baselines", fontweight="bold", fontsize=16)
fig.savefig(FIGURES / "11_nfl_three_year_epa_baselines.png", dpi=300, bbox_inches="tight")
plt.show()

### Figure 9 — Study-season QB performance under two reference windows

The **solid bar** compares a QB with the NFL in the exact same season. The **light bar** uses the smoothed window. Each bar reflects only starts included in our first-131-start sample. Consequently, isolated seasons with one or four starts can be unstable. 2026 is partial and uses a provisional trailing reference.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8.5), layout="constrained", sharey=True)
for ax, player in zip(axes, ORDER):
    subset = window_seasons.filter(pl.col("player_name") == player).sort("season")
    data = subset.to_dicts()
    years = [int(r["season"]) for r in data]
    annual = np.array([float(r["annual_relative_qb_epa_per_dropback"]) for r in data])
    smooth = np.array([float(r["three_year_relative_qb_epa_per_dropback"]) for r in data])
    idx = np.arange(len(years))
    w = 0.36
    ax.bar(idx-w/2, annual, w, color=COLORS[player], alpha=0.95,
           label="Annual-relative EPA/DB")
    ax.bar(idx+w/2, smooth, w, color=COLORS[player], alpha=0.29,
           edgecolor=COLORS[player], linewidth=1.0, label="Window-relative EPA/DB")
    ax.axhline(0, color=INK, linewidth=1.0)
    ax.set_xticks(idx, years)
    ax.set_title(player, loc="left", color=COLORS[player], fontweight="bold")
    ax.set_ylabel("QB relative EPA / dropback")
    ax.grid(axis="y", color=GRID, alpha=0.7)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    for j, row in enumerate(data):
        count = int(row["starts"])
        if count <= 5:
            y = max(float(annual[j]), float(smooth[j]))
            ax.annotate(f"{count} start{'s' if count > 1 else ''}",
                        (j, y), xytext=(0, 8), textcoords="offset points",
                        fontsize=8, color=SUBTLE, ha="center")
    if player == "Patrick Mahomes":
        ax.text(idx[-1], min(annual[-1], smooth[-1]) - 0.04,
                "partial", color=COLORS[player], fontsize=9, ha="center")
axes[0].legend(frameon=False, loc="upper left", ncol=2)
axes[-1].set_xlabel("NFL season (study starts only)")
fig.suptitle("Season-Relative vs Three-Year Relative QB EPA/DB", fontsize=16,
             fontweight="bold")
fig.savefig(FIGURES / "12_qb_season_vs_three_year_relative_epa.png", dpi=300, bbox_inches="tight")
plt.show()

### Figure 10 — Does smoothing change the 131-start comparison?

The paired estimates below use the **same 131 study starts and the same QB EPA numerator** under both league-baseline choices. The difference between bars comes *only* from the benchmark definition. Neither approach provides confidence intervals or controls for opponent/team effects.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5.1), layout="constrained")
ypos = np.arange(2)
bar_w = 0.34
for i, player in enumerate(ORDER):
    row = window_career.filter(pl.col("player_name") == player).row(0, named=True)
    annual = float(row["annual_relative_qb_epa_per_dropback"])
    smooth = float(row["three_year_relative_qb_epa_per_dropback"])
    ax.barh(i-bar_w/2, annual, height=bar_w, color=COLORS[player], label="Annual" if i == 0 else None)
    ax.barh(i+bar_w/2, smooth, height=bar_w, color=COLORS[player], alpha=0.36,
            edgecolor=COLORS[player], label="Three-year window" if i == 0 else None)
    ax.text(annual+0.004, i-bar_w/2, f"{annual:+.3f}", va="center", fontsize=11, fontweight="bold")
    ax.text(smooth+0.004, i+bar_w/2, f"{smooth:+.3f}", va="center", fontsize=11)
ax.set_yticks(ypos, ORDER)
ax.invert_yaxis()
ax.axvline(0, linewidth=1, color=INK)
ax.grid(axis="x", color=GRID)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.set_xlabel("Relative QB EPA per eligible dropback — first 131 starts")
ax.set_title("Does a Three-Year Reference Change the 131-Start Result?",
             fontsize=16, fontweight="bold")
ax.legend(frameon=False, ncol=2, loc="lower right")
fig.savefig(FIGURES / "13_qb_131_starts_annual_vs_three_year.png", dpi=300,
            bbox_inches="tight")
plt.show()

career_rows = {r["player_name"]: r for r in window_career.to_dicts()}
for method, key in (
    ("Annual", "annual_relative_qb_epa_per_dropback"),
    ("Three-year", "three_year_relative_qb_epa_per_dropback"),
):
    gap = career_rows["Patrick Mahomes"][key] - career_rows["Tom Brady"][key]
    print(f"{method} Mahomes − Brady relative EPA/DB gap: {gap:+.4f}")

### Exact values — study seasons and 131-start aggregates

Do not read precise results from the heights of the bars. The table below uses the processed Parquet values directly. **One-start and other small-sample years should not be interpreted as stable season-long performance.**

In [ ]:
# Compact table: actual annual-relative and three-year-relative EPA/DB.
rows = window_seasons.sort("player_name", "season").to_dicts()
output_lines = [
    "| Quarterback | Season | Study starts | Annual-relative EPA/DB | Three-year-relative EPA/DB | Change |",
    "|:--|--:|--:|--:|--:|--:|",
]
for player in ORDER:
    for r in rows:
        if r["player_name"] != player:
            continue
        yr = int(r["season"])
        mark = "†" if yr == 2026 else ""
        output_lines.append(
            f"| {player} | {yr}{mark} | {int(r['starts'])} | "
            f"{r['annual_relative_qb_epa_per_dropback']:+.4f} | "
            f"{r['three_year_relative_qb_epa_per_dropback']:+.4f} | "
            f"{r['three_year_minus_annual_relative_epa_per_dropback']:+.4f} |"
        )
output_lines.append("\n† 2026 uses a provisional trailing window with partial-season NFL data.")
display(Markdown("\n".join(output_lines)))

career_lines = [
    "| QB — first 131 starts | Dropbacks | Annual-relative EPA/DB | Three-year-relative EPA/DB | Change |",
    "|:--|--:|--:|--:|--:|",
]
for player in ORDER:
    r = career_rows[player]
    career_lines.append(
        f"| {player} | {int(r['usable_dropbacks']):,} | "
        f"{r['annual_relative_qb_epa_per_dropback']:+.4f} | "
        f"{r['three_year_relative_qb_epa_per_dropback']:+.4f} | "
        f"{r['three_year_minus_annual_relative_epa_per_dropback']:+.4f} |"
    )
display(Markdown("\n".join(career_lines)))

### Interpretation of the final three-year figures

The methodological questions above are answered in the **Final Research Report** below. Figure 8 documents the difference between annual and smoothed league references; Figure 9 shows how those references change individual season-relative EPA; Figure 10 measures the effect on the first-131-start comparison. The script outputs are the source of exact values, rather than rounded bar heights.


## 11. Final Research Report — Era-Adjusted QB Performance

*Can the Baby GOAT Surpass the GOAT? | Notebook 03 | First 131 regular-season starts*

### Executive Summary

This study compares Tom Brady and Patrick Mahomes over their first **131 NFL regular-season starts**, a deliberately matched career window in which both finished **100–31**. It asks a narrower question than who is the greatest quarterback: **which player generated more observed quarterback-attributed EPA per eligible dropback, relative to the NFL environments in which he played?**

**Primary finding:** Mahomes led on both absolute and season-relative efficiency. His observed absolute advantage was approximately **+0.073 QB EPA/dropback**; the difference narrowed to **+0.0381 EPA/dropback** after adjustment for season-specific league baselines. A three-year, dropback-weighted league reference produced a similar, slightly larger gap of **+0.0419 EPA/dropback**. Thus, **the direction of the observed difference is stable under the two league-reference definitions tested**, while its magnitude is considerably smaller than the unadjusted comparison.

The analysis also identifies contrasting career patterns: Mahomes entered his starting career with exceptionally high relative efficiency, while Brady's performance strengthened over time and culminated in an exceptional **2007** peak. These are descriptive findings about the selected games, **not estimates of QB-only causal impact, proof of statistical significance, or predictions of future performance**.

### 1. Research Design and Measurement

- **Sample:** 131 regular-season starts for each quarterback (262 QB-game records), excluding playoffs. In the EPA analysis, Brady contributes **4,607 usable eligible dropbacks** and Mahomes **5,381**.
- **Primary measure:** quarterback-attributed EPA (`qb_epa`) per eligible dropback, including passes, sacks, and tagged scrambles. Spikes, kneel-downs, and two-point attempts are excluded under the validated processing rules.
- **Annual reference:** the NFL's season-specific **dropback-weighted** mean QB EPA, calculated from the league's regular-season plays. The player's 131-start performance is aggregated by usable dropbacks rather than by taking an unweighted average of game rates.
- **Three-year reference:** the dropback-weighted NFL mean in a centered three-season window, with explicitly documented exceptions at study-era boundaries. This is a **smoothing sensitivity analysis**, not a replacement for the primary season-relative measure.
- **Peer ranking:** each player's **full-season** QB EPA/dropback percentile against qualified quarterbacks, with a minimum of 200 eligible dropbacks in the main specification. The qualification rule was also tested at 150 and 250 dropbacks.
- **Scope caveat:** the percentile comparison does not always cover exactly the same games as the 131-start comparison. The **2026 reference is partial through October 18, 2026**.

The following estimates are descriptive and conditional on the selected starts and the EPA model; differences in scheme, supporting cast, opposition and game situations remain present.

### 2. Headline Results — First 131 Starts

| Measure | Tom Brady | Patrick Mahomes | Mahomes − Brady |
|:--|--:|--:|--:|
| Usable dropbacks | 4,607 | 5,381 | — |
| Absolute QB EPA/dropback | ≈ +0.176 | ≈ +0.249 | ≈ **+0.073** |
| Era-matched NFL reference | ≈ +0.022 | ≈ +0.056 | — |
| **Annual-relative QB EPA/dropback** | **+0.1542** | **+0.1923** | **+0.0381** |
| **Three-year-relative QB EPA/dropback** | **+0.1526** | **+0.1945** | **+0.0419** |
| Change when using three-year reference | −0.0016 | +0.0022 | +0.0038 in the gap |

*The absolute and era-matched reference values are rounded from Figure 2; the annual- and three-year-relative values reproduce the four-decimal outputs supplied by the processing script. The last decimal of derived differences can change slightly if full-precision values are used.*

The season adjustment reduces the **observed absolute gap** from about +0.073 to +0.0381 EPA/dropback—approximately a **48% reduction in the descriptive difference**. This **does not mean** that historical era differences *caused* 48% of the original gap. The subtraction merely changes the reference against which each quarterback is evaluated.

For scale, the annual-relative gap equals about **3.81 expected points per 100 eligible dropbacks**; the smoothed-reference gap equals **4.19 per 100**. These are standardized exposure-rate differences, **not predicted scoring margins or expected additional wins**.

### 3. NFL Environment and Annual Relative Performance (Figures 1–3)

**League efficiency varied substantially within both eras**, rather than following a strictly increasing historical trend. For example, the NFL baseline was approximately **−0.0053** EPA/dropback in 2003, **+0.0312** in 2007, **+0.0740** in 2018, **+0.0938** in 2020, and **+0.0159** in 2023. Several Mahomes-era seasons were more efficient on average, but some were below Brady-era seasons. This variation justifies using **season-matched** rather than purely chronological assumptions about the passing environment.

Across the first 131 starts, **both QBs were well above their own era-matched NFL average**, with Mahomes retaining the larger relative aggregate. Their **season profiles**, however, were not uniform:

- **Brady's 2007 performance** exceeded the NFL baseline by approximately **+0.40 EPA/dropback**, the largest relative season segment visible for either player. His early seasons were more modest, while 2004–2007 and 2009 showed stronger relative performance. His negative 2008 bar covers **one study start**, so it is not evidence of a sustained season-long decline.
- **Mahomes showed particularly strong relative production in 2018–2022**, including peaks around **+0.30** in 2018 and **+0.28** in 2022. The subsequent 2023–2026 study segments were less elevated, but remained **above the league baseline** in the displayed sample. The 2017 segment includes **one start**; 2026 includes **five**.

Because these figures use only the starts in the research sample, a partial-season study segment should not be interpreted as the player's entire actual season. No causal attribution to receivers, coaching or defensive strategy can be made from these plots alone.

### 4. Career Evolution and Season Standardization (Figures 4–5)

The **cumulative, dropback-weighted relative EPA** trajectories reveal different shapes. Mahomes accumulated a large positive relative advantage early; as additional games were added, his cumulative average decreased but remained ahead at start 131. Brady began nearer the league reference and progressively increased his cumulative relative performance, particularly through his later study starts.

The **trailing 16-start** chart isolates shorter stretches of form. Mahomes's early and intermediate windows were highly productive, followed by weaker windows later in the comparison and some recovery. Brady's later windows—including the period associated with his 2007 peak—were substantially stronger than his earliest windows. Importantly, consecutive 16-start estimates share most of their underlying games: the curves are **not independent observations**, and apparent trends should not be treated as formal change-point or decline tests.

Season-standardized EPA divides season-relative QB EPA/dropback by the **NFL's play-level standard deviation of QB EPA** for that year. The visual pattern broadly resembles annual-relative EPA: Brady's **2007** season stands out, and Mahomes shows exceptional early-season segments. This is an **effect expressed in play-level dispersion units**, **not** a z-score over the distribution of quarterback means, a QB percentile, or evidence of statistical significance. The notebook also calculates relative positive-EPA play rates as a secondary check; numerical conclusions about that measure should come from its displayed table rather than from EPA-rate plots alone.

### 5. Qualified-Quarterback Peer Distributions (Figures 6–7)

Unlike the play-weighted league average, a qualified-QB percentile compares the **season-level efficiency of individual quarterbacks**. It answers the ranking question: *what fraction of the qualified QB cohort did this player outperform by QB EPA/dropback?*

| Brady study-era season | Percentile | Mahomes study-era season | Percentile |
|:--|--:|:--|--:|
| 2001 | 56.7%† | 2017 | N/Q |
| 2002 | 42.9% | 2018 | **100.0%** |
| 2003 | 67.6% | 2019 | 97.0% |
| 2004 | 82.9% | 2020 | 97.1% |
| 2005 | 88.9% | 2021 | 97.1% |
| 2006 | 86.1% | 2022 | **100.0%** |
| 2007 | **100.0%** | 2023 | 86.8% |
| 2008 | N/Q | 2024 | 76.9% |
| 2009 | 87.9% | 2025 | 78.4% |
| 2010 | **100.0%†** | 2026 | 70.6%‡ |

*Values are rounded to one decimal for readability. N/Q means that the player did not satisfy the specified minimum. † Full-season rankings include QB plays outside the selected starts (notably Brady in 2001 and 2010). ‡ 2026 is provisional and based on the truncated league period.*

Brady's peer ranking progressed from the middle or lower-middle of the qualified distribution in his earliest seasons to the **top-ranked qualified position in 2007**. Mahomes was top-ranked in **2018 and 2022** and near the top through 2021, although his percentile declined in 2023–2026. A lower percentile does **not** mean below-average absolute EPA, nor does a percentile of 100 imply perfect performance or a fixed magnitude of superiority over second place.

**Sensitivity to the qualification threshold.** Recomputing percentiles with minimums of **150, 200 and 250** dropbacks yielded a mean within-season range of approximately **3.9 percentage points for Brady** and **1.5 points for Mahomes** across their nine qualified seasons. Brady's largest range was **11.3 pp in 2002**; Mahomes's was **4.3 pp in 2023**. The top placements for **Brady 2007** and **Mahomes 2018 and 2022** remained at **100.0%** under all three thresholds. Hence the headline peer-ranking patterns are reasonably robust to these *specific* qualification choices, though the threshold affects some early Brady years more strongly.

### 6. Three-Year NFL Baselines — What Changed? (Figures 8–10)

#### Figure 8 — Annual vs Three-Year League Environment

The smoothed NFL reference is **less volatile** than a single-year baseline. That is the intended behavior of averaging adjacent seasons; it should **not** be interpreted as evidence that league efficiency itself was stable.

Examples from the computed baselines:

| NFL season | Annual baseline | Three-year reference | What changes? |
|:--|--:|--:|:--|
| 2003 | −0.0053 | +0.0223 | A low annual baseline is pulled upward by neighboring years. |
| 2004 | +0.0438 | +0.0204 | A comparatively strong annual baseline is pulled downward. |
| 2018 | +0.0740 | +0.0520 | The smoothed reference is less demanding. |
| 2020 | +0.0938 | +0.0689 | A particularly high annual league baseline is softened. |
| 2023 | +0.0159 | +0.0385 | A low annual baseline is pulled upward. |
| 2026‡ | +0.0702 | +0.0625 | Provisional reference mixes 2024–2025 with part of 2026. |

*‡ 2026 is partial through October 18, 2026. Values rounded to four decimals.*

The comparison does **not** join the disconnected study eras (2010 and 2017). Most windows use the previous, current and next seasons weighted by their NFL dropback counts. At the edges, **2001, 2010, 2017 and 2025 use two full seasons**; **2026 uses 2024 and 2025 plus the observed partial 2026 season**. These exceptions mean this is not a literal three-*complete*-season reference for every row. Centered windows use future-season information and are appropriate only for **retrospective description**, not real-time forecasting.

#### Figure 9 — Season-Level Effects of Changing the Baseline

In this comparison, the quarterback's EPA numerator and sample do **not** change. Only the **league reference subtracted** from it changes. Consequently, when the smoothed league baseline is *lower* than the annual baseline, the QB's relative EPA rises; when it is *higher*, relative EPA falls.

- **Brady:** in **2003**, replacing the unusually low annual baseline with a higher three-year reference reduces his relative EPA. In **2004**, replacing a higher annual baseline with a lower smoothed reference increases it. His standout **2007** season remains exceptional under both calculations. The 2008 and 2010 segments contain only **one** and **four** selected starts, respectively.
- **Mahomes:** **2018 and 2020** look somewhat better relative to the smoothed reference because those years' annual league baselines were higher than their windows. In **2023**, the opposite happens: the higher smoothed baseline reduces his estimated relative advantage. His **2022** peak remains strong under both measures. The 2026 bar is provisional and covers **five** study starts.

These shifts show why a one-year comparison may emphasize a particularly favorable or unfavorable *league-wide* season. They do **not**, by themselves, show changes in quarterback skill.

#### Figure 10 — The Aggregate 131-Start Answer

With the **annual** baseline, the observed Mahomes–Brady gap is **+0.0381 EPA/dropback**. With the **three-year** reference, the gap is **+0.0419**. This is a change of **+0.0038 EPA/dropback** (about **10% relative to the annual-gap magnitude**), and **the ranking does not reverse**.

Brady moves from **+0.1542** to **+0.1526**, while Mahomes moves from **+0.1923** to **+0.1945**. The overall finding therefore appears **not highly sensitive to these two particular baseline windows**. This is evidence of *reference-definition robustness*, not a test of statistical significance and not an independent replication of QB EPA.

### 7. Main Findings and Strength of Evidence

1. **Observed advantage:** Mahomes produced higher absolute and era-relative QB EPA/dropback across the selected 131 starts; the annual-adjusted advantage was about **+0.0381 EPA/dropback**.
2. **Era adjustment matters:** subtracting season-specific NFL baselines reduced the observed gap by approximately **48%** compared with the rounded absolute difference.
3. **Reference smoothing does not overturn the result:** the three-year window gave Mahomes a **+0.0419** advantage, close to the annual estimate.
4. **Different trajectories:** Mahomes's early starting years were unusually efficient; Brady's relative efficiency improved across his early starts and peaked sharply in **2007**. The relative EPA curves are descriptive time-series summaries, not causal trend estimates.
5. **Peer-group evidence is consistent with those patterns:** Brady was top-ranked among qualified QBs in **2007**; Mahomes was top-ranked in **2018 and 2022**. These positions remained top-ranked under all three tested minimum-volume thresholds.
6. **A QB's season percentile and EPA relative to the league are not interchangeable:** percentiles capture order among qualifying players; EPA gaps capture a magnitude relative to a *play-weighted* league reference.

### 8. Limitations and Threats to Interpretation

- **Observational, not QB-only causal:** QB EPA retains effects from receivers, offensive line, coaching, opponent defenses, game situations and EPA-model conventions. The study does not isolate individual quarterback talent.
- **No formal uncertainty assessment:** no game-level clustered or season-aware bootstrap intervals or hypothesis tests have yet been reported. A positive observed gap must not be presented as statistically significant.
- **Selected endpoints:** choosing both samples because they reached **100 wins in 131 starts** conditions on a team outcome. The comparison excludes playoffs and may omit useful performance information outside that window.
- **Exposure differs:** Brady and Mahomes contributed **4,607** and **5,381** usable dropbacks, respectively. Per-dropback comparisons adjust for exposure in the denominator, but not for differing opponents or game contexts.
- **Incomplete/unequal seasons:** the study's season bars may contain few starts (Brady 2008: 1; Brady 2010: 4; Mahomes 2017: 1; Mahomes 2026: 5). Full-season peer rankings can include QB plays outside those selected starts; 2026 has an incomplete NFL reference.
- **Historical play classification:** historically tagged scrambles missing the original dropback flag were restored for the EPA definition; untagged QB runs were *not* automatically classified as scrambles. Historical classification choices remain a source dependency.
- **Known missing observation:** one eligible 2019 play (`2019_01_PIT_NE`, play 4166) had missing QB EPA and was excluded from the corresponding league EPA denominator rather than imputed as zero.
- **Reference construction:** the league mean includes the study QBs' own plays, is weighted by dropbacks (not equally by quarterbacks), and is not an opponent-adjusted or qualified-QB-only baseline.
- **Window design:** centered three-year means incorporate adjacent years, including future years, and have unequal edge-year scopes. The partial 2026 observation may change with additional games. Smoothing cannot remove underlying team context or establish a causal era effect.
- **Standardized score interpretation:** dividing by the standard deviation of individual NFL plays is not a percentile among QBs or a standard error of the quarterback's mean.

### 9. Conclusion and Next Research Steps

**Within these first 131 regular-season starts, Mahomes had the higher observed era-relative QB EPA/dropback, and that lead persisted when the league reference was smoothed over neighboring seasons.** However, the gap was notably smaller than the absolute EPA difference. **Brady's extraordinary 2007 peak and the contrasting career trajectories remain important complementary findings.**

This notebook supports a conclusion about **observed, context-referenced quarterback efficiency in a fixed historical sample**. It does **not** settle who was the better individual player or which quarterback will ultimately have the greater career.

Before treating the measured gap as a strong comparative claim, future work should: **(1)** estimate uncertainty with game-level and preferably season-aware resampling; **(2)** test sensitivity to excluding high-leverage seasons such as Brady 2007 and Mahomes 2018/2022, and to excluding partial 2026; **(3)** examine alternate baselines, including QB-qualified and leave-study-QB-out references; and **(4)** evaluate team and opponent context. These are natural extensions beyond this completed **descriptive era-adjusted analysis**.


## 12. Outputs and reproducibility

Figures written automatically to `reports/figures/`:

- `04_nfl_era_baselines.png`
- `05_qb_vs_era_baseline_131.png`
- `06_season_relative_qb_epa.png`
- `07_era_relative_career_evolution.png`
- `08_season_normalized_qb_epa.png`

Existing validated tables under `data/processed/` are never overwritten. To regenerate plots, rerun this notebook from the first cell after rebuilding its upstream processed tables. The final report above is based on the observed figures and the numerical outputs supplied from the executed pipelines. Re-run the notebook after refreshing source datasets.

**Data provenance:** `docs/era_adjustment_baseline_decisions.md` and `docs/era_adjusted_qb_methodology.md`.

Additional figures (after running the qualified-QB ranking pipeline):

- `09_qualified_qb_season_distributions.png`
- `10_qualified_qb_percentiles.png`


The **qualified QB percentile comparison table** appears beneath Figure 7 and is generated directly from `data/processed/study_qb_qualified_percentiles.parquet` on each execution.


The **qualification-threshold sensitivity summary** appears after the percentile table and writes `reports/validation/qualified_qb_sensitivity_summary.csv` when executed. The raw all-QB report remains `reports/validation/qualified_qb_threshold_sensitivity.csv`.


**Three-year window sensitivity** (after running `scripts/build_three_year_relative_epa.py`):

- `11_nfl_three_year_epa_baselines.png`
- `12_qb_season_vs_three_year_relative_epa.png`
- `13_qb_131_starts_annual_vs_three_year.png`

The league-window table, per-game values, study-season summaries, and 131-start aggregates are saved as distinct Parquet/CSV files. See `docs/three_year_relative_epa_methodology.md` for the explicitly defined edge-year and partial-year policies.
